# Operation, temperature and time extraction — training-set characterisation and data preparation

**Input:** `0_Labeled_Synthesis_Sep_24_2026.xlsx`, sheet `1_Training_Set_First12`.

**Section 1 — Training-set composition.**
Operation-type frequencies across the training sheet, the in-scope / out-of-scope
split, and the per-year distribution of the most frequent operations. Establishes
which operations the framework is trained on.

**Section 2 — Reporting practice and standardisation.**
Calcination, the most frequent operation, is used as the representative case:
non-numeric parameter values, unit notation variants for temperature, time and
heating rate, indirect heating-rate prose, and variation in the operation verb.

**Section 3 — Data preparation.**
Writes the training sheet consumed by the Snowball training notebook.

# Section 1 — Training-set composition


In [ ]:
# 1.1 — Load the labelled training sheet

import re
import pandas as pd

INPUT_XLSX = "0_Labeled_Synthesis_Sep_24_2026.xlsx"
SHEETS = ["1_Training_Set_First12"]

raw = pd.read_excel(INPUT_XLSX, sheet_name=SHEETS, dtype=str)


## 1.2 Operation-type frequency


In [ ]:
# 1.2 — Operation-type frequency over every annotated event

import matplotlib.pyplot as plt
from pathlib import Path

VIS_DIR = Path("full_table_visualizations")
VIS_DIR.mkdir(exist_ok=True)

# Combine all original sheets without changing `raw`
full_original = []

for sheet_name, df0 in raw.items():
    temp = df0.copy()
    temp.columns = [str(c).strip() for c in temp.columns]
    temp.insert(0, "source_sheet", sheet_name)
    full_original.append(temp)

full_original = pd.concat(full_original, ignore_index=True)

# Check required column
if "operation_type" not in full_original.columns:
    raise KeyError("Column 'operation_type' was not found in the original table.")

# Clean operation_type values for analysis only
operation_type_clean = (
    full_original["operation_type"]
    .fillna("missing")
    .astype(str)
    .str.strip()
    .replace("", "missing")
    .str.lower()
)

# Frequency table
operation_counts = (
    operation_type_clean
    .value_counts()
    .rename_axis("operation_type")
    .reset_index(name="frequency")
)

operation_counts["percentage"] = (
    operation_counts["frequency"] / operation_counts["frequency"].sum() * 100
)

print("\nOperation type frequency in the FULL original table:")
print(operation_counts)

operation_counts.to_csv(
    VIS_DIR / "operation_type_frequency_full_original.csv",
    index=False
)

## 1.3 Operation-type distribution and per-year counts


In [ ]:
from matplotlib.ticker import MaxNLocator

# 1.3 — (a) operation-type distribution, (b) per-year counts for the top types.
# Operation types with RARE_THRESHOLD events or fewer are out of scope and are
# excluded from the figure, so "other (included)" covers only in-scope types.

TOP_N = 4
RARE_THRESHOLD = 6

operation_counts_clean = (
    operation_counts
    .loc[operation_counts["operation_type"].astype(str).str.lower().ne("missing")]
    .copy()
    .sort_values("frequency", ascending=False)
)

if operation_counts_clean.empty:
    raise ValueError("No operation types available after removing 'missing'.")

in_scope_counts = operation_counts_clean[
    operation_counts_clean["frequency"] > RARE_THRESHOLD
].copy()

out_of_scope_counts = operation_counts_clean[
    operation_counts_clean["frequency"] <= RARE_THRESHOLD
].copy()

print(
    f"In scope (n > {RARE_THRESHOLD}):     "
    f"{len(in_scope_counts):3d} types, "
    f"{int(in_scope_counts['frequency'].sum()):4d} events"
)
print(
    f"Out of scope (n <= {RARE_THRESHOLD}): "
    f"{len(out_of_scope_counts):3d} types, "
    f"{int(out_of_scope_counts['frequency'].sum()):4d} events  "
    f"(excluded from this figure)"
)

top4_operations = in_scope_counts.head(TOP_N).copy()
top4_order = top4_operations["operation_type"].tolist()

# Overall distribution: top N + the remaining IN-SCOPE types
overall_distribution = top4_operations[["operation_type", "frequency"]].copy()

if len(in_scope_counts) > TOP_N:
    remaining_in_scope = in_scope_counts.iloc[TOP_N:]

    print(
        f"'other (included)' = {len(remaining_in_scope)} in-scope types: "
        + ", ".join(remaining_in_scope["operation_type"].astype(str))
    )

    other_row = pd.DataFrame({
        "operation_type": ["other (included)"],
        "frequency": [remaining_in_scope["frequency"].sum()]
    })

    overall_distribution = pd.concat(
        [overall_distribution, other_row],
        ignore_index=True
    )

overall_distribution["percentage"] = (
    overall_distribution["frequency"]
    / overall_distribution["frequency"].sum()
    * 100
)

overall_bar_data = overall_distribution.sort_values("frequency")


# -------------------------
# Per-year analysis for top operation types
# -------------------------

if "year" not in full_original.columns:
    raise KeyError("Column 'year' was not found in the original table.")

year_clean = (
    full_original["year"]
    .fillna("")
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

analysis_df = full_original.assign(
    operation_type_clean=operation_type_clean,
    year_clean=year_clean
)

analysis_df = analysis_df[
    analysis_df["operation_type_clean"].isin(top4_order)
    & analysis_df["year_clean"].notna()
].copy()

analysis_df["year_clean"] = analysis_df["year_clean"].astype(int)

operation_by_year = (
    analysis_df
    .groupby(["year_clean", "operation_type_clean"])
    .size()
    .reset_index(name="frequency")
    .sort_values(["year_clean", "frequency"], ascending=[True, False])
)

operation_by_year.to_csv(
    VIS_DIR / "operation_type_top4_frequency_by_year.csv",
    index=False
)

operation_by_year_pivot = (
    operation_by_year
    .pivot(
        index="year_clean",
        columns="operation_type_clean",
        values="frequency"
    )
    .reindex(columns=top4_order)
    .fillna(0)
    .sort_index()
    .astype(int)
)

if operation_by_year_pivot.empty:
    raise ValueError("No valid per-year data found for the top operation types.")


# -------------------------
# Report-ready styling
# -------------------------

color_map = {
    "calcination": "#0072B2",
    "drying": "#E69F00",
    "stirring": "#009E73",
    "heating": "#D55E00",
    "aging": "#CC79A7",
    "other": "#999999",
    "other (included)": "#999999"
}

fallback_cmap = plt.get_cmap("tab20")

all_labels = list(dict.fromkeys(
    overall_distribution["operation_type"].tolist() + top4_order
))

fallback_color_map = {
    label: fallback_cmap(i % fallback_cmap.N)
    for i, label in enumerate(all_labels)
}

def get_operation_color(label):
    return color_map.get(
        str(label).lower(),
        fallback_color_map.get(label, "#999999")
    )


with plt.rc_context({
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
    "legend.fontsize": 10
}):

    fig, (ax_total, ax_year) = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(13, 5.5),
        gridspec_kw={"width_ratios": [1.1, 2.4]},
        constrained_layout=True
    )

    # -------------------------
    # (a) Overall distribution
    # -------------------------

    ax_total.barh(
        overall_bar_data["operation_type"],
        overall_bar_data["frequency"],
        color=[
            get_operation_color(op)
            for op in overall_bar_data["operation_type"]
        ],
        edgecolor="white",
        linewidth=0.8
    )

    max_total = max(overall_bar_data["frequency"].max(), 1)
    ax_total.set_xlim(0, max_total * 1.28)

    for y_pos, (_, row) in enumerate(overall_bar_data.iterrows()):
        ax_total.text(
            row["frequency"] + max_total * 0.025,
            y_pos,
            f'{row["frequency"]:,.0f} ({row["percentage"]:.1f}%)',
            va="center",
            ha="left",
            fontsize=10
        )

    ax_total.set_xlabel("Number of events")
    ax_total.set_ylabel("")
    ax_total.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax_total.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.35)
    ax_total.set_axisbelow(True)

    ax_total.text(
        -0.18, 1.06,
        "(a)",
        transform=ax_total.transAxes,
        fontsize=14,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False
    )


    # -------------------------
    # (b) Per-year distribution
    # -------------------------

    operation_by_year_pivot.plot(
        kind="bar",
        ax=ax_year,
        width=0.82,
        color=[
            get_operation_color(op)
            for op in operation_by_year_pivot.columns
        ],
        edgecolor="white",
        linewidth=0.5
    )

    ax_year.set_xlabel("Year")
    ax_year.set_ylabel("Number of events")
    ax_year.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax_year.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.35)
    ax_year.set_axisbelow(True)

    for tick in ax_year.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha("right")

    ax_year.legend(
        title=None,
        frameon=False,
        ncol=min(len(operation_by_year_pivot.columns), 4),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.16)
    )

    ax_year.text(
        -0.08, 1.06,
        "(b)",
        transform=ax_year.transAxes,
        fontsize=14,
        fontweight="bold",
        va="bottom",
        ha="left",
        clip_on=False
    )

    for ax in (ax_total, ax_year):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.savefig(
        VIS_DIR / "operation_type_combined_total_and_by_year.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# Section 2 — Reporting practice and standardisation

Calcination is the most frequent operation in the training sheet and is used here as
the representative case: it has enough events to show each reporting problem without
mixing operations that report their parameters differently.


In [ ]:
# 2.1 — Build the calcination-only subset

import pandas as pd

INPUT_XLSX = "0_Labeled_Synthesis_Sep_24_2026.xlsx"
OUTPUT_XLSX = "calcination_rows_only.xlsx"
SHEET_NAME = "1_Training_Set_First12"

FILTER_COL = "operation_type"
FILTER_VALUE = "calcination"

GROUP_COLS = ["title", "paragraph_text"]


def filter_calcination_rows(df):
    df = df.copy()

    # Clean column names
    df.columns = [str(c).strip() for c in df.columns]

    # Forward-fill title and paragraph_text
    existing_group_cols = [c for c in GROUP_COLS if c in df.columns]
    if existing_group_cols:
        df[existing_group_cols] = (
            df[existing_group_cols]
            .replace(r"^\s*$", pd.NA, regex=True)
            .ffill()
        )

    # Keep only calcination rows
    df = df[
        df[FILTER_COL]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(FILTER_VALUE)
    ].copy()

    return df


# Read the labelled sheet
df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Filter calcination rows
calcination_df = filter_calcination_rows(df)

# Save one output sheet only
calcination_df.to_excel(
    OUTPUT_XLSX,
    sheet_name="Training_calcination_only",
    index=False
)

print(f"Saved: {OUTPUT_XLSX}")
print(f"Columns saved: {len(calcination_df.columns)}")
print(f"Calcination rows: {len(calcination_df)}")


## 2.2 Parameter values


In [ ]:
# 2.2 — Load the calcination subset


from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

FILE_STEM = "calcination_rows_only"
SHEET_NAME = "Training_calcination_only"

candidate_files = [
    Path(f"{FILE_STEM}.xlsx"),
    Path(f"{FILE_STEM}.xls"),
    Path(f"{FILE_STEM}.xlsm"),
    Path(FILE_STEM),
]

for p in candidate_files:
    if p.exists():
        FILE_PATH = p
        break
else:
    raise FileNotFoundError(
        "Could not find calcination_rows_only.xlsx / .xls / .xlsm in the current folder."
    )

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Clean column names
df.columns = df.columns.astype(str).str.strip()

# Excel row number: header is row 1, first data row is row 2
df["_excel_row_number"] = np.arange(len(df)) + 2

required_cols = [
    "temperature_value",
    "temperature_unit",
    "time_value",
    "time_unit",
    
    "ramp_rate_text_span",	
    "ramp_value",
    "ramp_unit",

    
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected columns: {missing_cols}")

print(f"Loaded file: {FILE_PATH}")
print(f"Sheet: {SHEET_NAME}")
print(f"Rows: {len(df)}")
display(df.head())

In [ ]:
# 2.2 — Numeric cleaning helpers


NUMERIC_PATTERN = re.compile(
    r"^\s*[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?\s*$"
)

def clean_text_cell(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def strict_numeric_series(series):
    """
    Converts only simple numeric values to float.
    Text such as 'same heating rate', '500-600', '~600', or '600 °C'
    is treated as irregular and excluded from plots.
    """
    text = series.map(clean_text_cell)
    text = (
        text.str.replace("\u2212", "-", regex=False)  # unicode minus
            .str.replace(",", "", regex=False)        # 1,000 -> 1000
            .str.strip()
    )

    nonempty_mask = text.ne("")
    numeric_mask = text.str.match(NUMERIC_PATTERN)

    numeric = pd.Series(np.nan, index=series.index, dtype="float64")
    numeric.loc[numeric_mask] = pd.to_numeric(text.loc[numeric_mask], errors="coerce")

    irregular_mask = nonempty_mask & ~numeric_mask

    return numeric, irregular_mask, text


df["_ramp_numeric"], ramp_value_irregular_mask, df["_ramp_value_text"] = strict_numeric_series(
    df["ramp_value"]
)


In [ ]:
# 2.3 — Temperature and time values reported without a number

id_cols = [c for c in ["_excel_row_number", "ID"] if c in df.columns]

required_span_cols = ["temperature_text_span", "time_text_span"]
missing_span_cols = [c for c in required_span_cols if c not in df.columns]
if missing_span_cols:
    raise KeyError(f"Missing required column(s): {missing_span_cols}")

# Irregular: the text span exists but contains no numeric value.
number_pattern = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"

def make_irregular_text_span_mask(column_name):
    text = df[column_name].astype("string").str.strip()
    has_text = text.notna() & text.ne("")
    has_number = text.str.contains(number_pattern, regex=True, na=False)
    return has_text & ~has_number

temperature_irregular_mask = make_irregular_text_span_mask("temperature_text_span")
time_irregular_mask = make_irregular_text_span_mask("time_text_span")

irregular_reports = []

temp_irregular = df.loc[
    temperature_irregular_mask,
    id_cols + ["temperature_text_span"]
].copy()

if not temp_irregular.empty:
    temp_irregular.insert(0, "parameter", "temperature_text_span")
    temp_irregular = temp_irregular.rename(
        columns={
            "temperature_text_span": "reported_text_span"
        }
    )
    irregular_reports.append(temp_irregular)

time_irregular = df.loc[
    time_irregular_mask,
    id_cols + ["time_text_span"]
].copy()

if not time_irregular.empty:
    time_irregular.insert(0, "parameter", "time_text_span")
    time_irregular = time_irregular.rename(
        columns={
            "time_text_span": "reported_text_span"
        }
    )
    irregular_reports.append(time_irregular)

if irregular_reports:
    irregular_value_report = pd.concat(irregular_reports, ignore_index=True)
    print("Irregular non-numeric text-span values excluded from the value-range plots:")
    display(irregular_value_report)
else:
    print("No irregular non-numeric entries found in temperature_text_span or time_text_span.")

In [ ]:
# 2.4 — Temperature and time unit frequency


def unit_frequency_table(series, missing_label="(missing)"):
    cleaned = series.map(clean_text_cell)
    cleaned = cleaned.mask(cleaned.eq(""), missing_label)

    return (
        cleaned.value_counts(dropna=False)
        .rename_axis("unit_as_reported")
        .reset_index(name="frequency")
    )

temperature_unit_counts = unit_frequency_table(df["temperature_unit"])
time_unit_counts = unit_frequency_table(df["time_unit"])

print("Temperature unit frequencies:")
display(temperature_unit_counts)

print("Time unit frequencies:")
display(time_unit_counts)

In [ ]:
# 2.4 — Temperature unit frequency plot

fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(temperature_unit_counts))))

ax.barh(
    temperature_unit_counts["unit_as_reported"].astype(str),
    temperature_unit_counts["frequency"]
)

ax.set_title("Temperature Unit Frequency")
ax.set_xlabel("Frequency")
ax.set_ylabel("temperature_unit as reported")
ax.invert_yaxis()

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

In [ ]:
# 2.4 — Time unit frequency plot

fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(time_unit_counts))))

ax.barh(
    time_unit_counts["unit_as_reported"].astype(str),
    time_unit_counts["frequency"]
)

ax.set_title("Time Unit Frequency")
ax.set_xlabel("Frequency")
ax.set_ylabel("time_unit as reported")
ax.invert_yaxis()

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()

## 2.5 Heating rate

Reported for the standardisation analysis only; heating rate is not an extraction target.


In [ ]:
# 2.5 — Heating-rate reporting summary

ramp_text = df["ramp_rate_text_span"].map(clean_text_cell)
ramp_value_text = df["_ramp_value_text"]
ramp_unit_text = df["ramp_unit"].map(clean_text_cell)

has_any_ramp_information = (
    ramp_text.ne("") |
    ramp_value_text.ne("") |
    ramp_unit_text.ne("")
)

has_direct_numeric_ramp_value = df["_ramp_numeric"].notna()

has_ramp_text_but_no_direct_numeric_value = (
    ramp_text.ne("") &
    ~has_direct_numeric_ramp_value
)

summary = pd.DataFrame({
    "metric": [
        "Total rows",
        "Rows with any heating-rate information",
        "Rows with direct numeric ramp_value",
        "Rows with ramp information but no direct numeric ramp_value",
        "Rows with no heating-rate information"
    ],
    "count": [
        len(df),
        has_any_ramp_information.sum(),
        has_direct_numeric_ramp_value.sum(),
        has_ramp_text_but_no_direct_numeric_value.sum(),
        (~has_any_ramp_information).sum()
    ]
})

display(summary)


### 2.5.1 Unit notation variants


In [ ]:
# 2.5.1 — Heating-rate unit notation variants

import re
import pandas as pd

ramp_text = df["ramp_rate_text_span"].map(clean_text_cell)
ramp_unit_text = df["ramp_unit"].map(clean_text_cell)

# ramp_unit is already normalised, so the variants are read from the raw
# ramp_rate_text_span instead.
raw_ramp_text_scope = ramp_text.ne("")

UNIT_VARIANT_RE = re.compile(
    r"""
    (?P<unit>
        (?:
            °\s*C
            | º\s*C
            | ℃
            | o\s*C
            | deg(?:ree)?s?\s*C
            | (?<![A-Za-z])C(?![A-Za-z])
            | (?<![A-Za-z])K(?![A-Za-z])
        )
        \s*
        (?:
            [/⁄]\s*min
            |
            (?:
                [.\u00B7⋅∙•]\s*
            )?
            min
            \s*
            (?:
                [-−–—]\s*1
                | \^\s*[-−–—]?\s*1
                | ⁻\s*¹
                | ⁻¹
                | ⁻\s*1
            )
        )
    )
    """,
    flags=re.VERBOSE | re.IGNORECASE
)


def _clean_for_unit_variant_search(value):
    s = clean_text_cell(value)
    return re.sub(r"\s+", " ", s).strip()


def extract_heating_rate_unit_variants(value):
    s = _clean_for_unit_variant_search(value)

    if not s:
        return []

    variants = [
        re.sub(r"\s+", " ", match.group("unit")).strip()
        for match in UNIT_VARIANT_RE.finditer(s)
    ]

    # Count the same variant only once per row.
    return list(dict.fromkeys(variants))


def canonical_heating_rate_unit(unit_variant):
    unit_variant = clean_text_cell(unit_variant)

    if re.match(r"(?i)^K\b", unit_variant):
        return "K/min"

    if re.match(
        r"(?i)^(?:°\s*C|º\s*C|℃|o\s*C|deg(?:ree)?s?\s*C|C\b)",
        unit_variant
    ):
        return "°C/min"

    return unit_variant


def heating_rate_unit_notation_style(unit_variant):
    if "/" in unit_variant or "⁄" in unit_variant:
        return "slash per minute"

    if "." in unit_variant:
        return "period inverse minute"

    if any(dot in unit_variant for dot in ["·", "⋅", "∙", "•"]):
        return "dot inverse minute"

    if re.search(
        r"(?:[-−–—]\s*1|\^\s*[-−–—]?\s*1|⁻\s*¹|⁻¹|⁻\s*1)",
        unit_variant
    ):
        return "space/no-separator inverse minute"

    return "other"


records = []

for row_index in df.index[raw_ramp_text_scope]:
    reported_text = _clean_for_unit_variant_search(ramp_text.loc[row_index])
    variants = extract_heating_rate_unit_variants(reported_text)

    for variant in variants:
        records.append({
            "row_index": row_index,
            "canonical_unit": canonical_heating_rate_unit(variant),
            "unit_variant_as_reported": variant,
            "notation_style": heating_rate_unit_notation_style(variant),
            "example_rate_text": reported_text
        })


unit_variant_rows = pd.DataFrame(records)

if unit_variant_rows.empty:
    ramp_unit_variant_counts = pd.DataFrame(
        columns=[
            "canonical_unit",
            "unit_variant_as_reported",
            "notation_style",
            "frequency",
            "example_rate_text"
        ]
    )
else:
    ramp_unit_variant_counts = (
        unit_variant_rows
        .groupby(
            ["canonical_unit", "unit_variant_as_reported", "notation_style"],
            dropna=False,
            as_index=False
        )
        .agg(
            frequency=("row_index", "nunique"),
            example_rate_text=("example_rate_text", "first")
        )
        .sort_values(
            ["canonical_unit", "frequency", "unit_variant_as_reported"],
            ascending=[True, False, True]
        )
        .reset_index(drop=True)
    )


print("Raw heating-rate unit notation variants found in ramp_rate_text_span:")
display(ramp_unit_variant_counts)


# Rows with ramp text but no unit notation detected
extracted_variants_per_row = ramp_text.map(extract_heating_rate_unit_variants)
has_extracted_unit_variant = extracted_variants_per_row.map(bool)

unmatched_ramp_texts = (
    df.loc[
        raw_ramp_text_scope & ~has_extracted_unit_variant,
        ["ramp_rate_text_span", "ramp_unit", "_ramp_value_text", "_ramp_numeric"]
    ]
    .copy()
)

unmatched_ramp_texts["cleaned_ramp_rate_text_span"] = unmatched_ramp_texts[
    "ramp_rate_text_span"
].map(clean_text_cell)

print("Rows with ramp text but no raw heating-rate unit notation detected:")
display(unmatched_ramp_texts)

In [ ]:
# 2.5.2 — Heating rates given as indirect or non-numeric text

ramp_irregular_mask = (
    has_ramp_text_but_no_direct_numeric_value |
    ramp_value_irregular_mask
)

ramp_report_cols = id_cols + [
    "ramp_rate_text_span",
    "ramp_value",
    "ramp_unit"
]

ramp_irregular_report = df.loc[
    ramp_irregular_mask,
    ramp_report_cols
].copy()

if ramp_irregular_report.empty:
    print("No indirect or irregular heating-rate text found.")
else:
    print(
        "Rows where heating rate appears as indirect text or non-numeric value "
        "instead of a directly reported numeric ramp_value:"
    )
    display(ramp_irregular_report)

## 2.6 Operation verb variation


In [ ]:
# 2.6 — Operation verb / phrase frequency

operation_col = "operation_verb_text_span"

if operation_col not in df.columns:
    raise KeyError(f"Missing expected column: {operation_col}")

operation_text = df[operation_col].map(clean_text_cell)
operation_text = operation_text[operation_text.ne("")]

if operation_text.empty:
    print("No operation phrases found in operation_verb_text_span.")
else:
    operation_phrases = (
        operation_text
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    operation_phrase_counts = (
        operation_phrases.value_counts()
        .rename_axis("operation_phrase")
        .reset_index(name="frequency")
    )

    display(operation_phrase_counts)

    plot_df = operation_phrase_counts.sort_values("frequency", ascending=True)

    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(plot_df))))

    ax.barh(
        plot_df["operation_phrase"],
        plot_df["frequency"]
    )

    ax.set_title("Frequency of Operation Verb / Operation Phrases")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("operation_verb_text_span as reported")

    for container in ax.containers:
        ax.bar_label(container)

    plt.tight_layout()
    plt.show()


In [ ]:
# 2.6 — Operation verb distribution: top 3 phrases and "Other"

import textwrap
import matplotlib.pyplot as plt
import pandas as pd


counts = (
    operation_phrase_counts
    .set_index("operation_phrase")["frequency"]
    .sort_values(ascending=False)
)


top3 = counts.head(3)
other_count = counts.iloc[3:].sum()

pie_counts = pd.concat([
    top3,
    pd.Series({"Other": other_count})
])

labels = pie_counts.index.tolist()
values = pie_counts.values
total = values.sum()

# Same 4 colors as the report bar plot:
# blue, yellow, green, orange
colors = [
    "#0077B6",  # blue
    "#E69F00",  # yellow
    "#009E73",  # green
    "#D55E00",  # orange
]

def autopct_with_labels(labels, values):
    label_iter = iter(labels)
    total = sum(values)

    def inner(pct):
        label = next(label_iter)
        count = int(round(pct * total / 100))

        wrapped_label = "\n".join(textwrap.wrap(str(label), width=14))

        return f"{wrapped_label}\n{pct:.1f}%\n(n={count})"

    return inner

fig, ax = plt.subplots(figsize=(7.2, 7.2))

wedges, texts, autotexts = ax.pie(
    values,
    labels=None,
    colors=colors,
    autopct=autopct_with_labels(labels, values),
    startangle=90,
    counterclock=False,
    pctdistance=0.62,
    wedgeprops={
        "edgecolor": "white",
        "linewidth": 2.0
    },
    textprops={
        "ha": "center",
        "va": "center",
        "fontsize": 13,
        "fontweight": "bold"
    }
)

# Adjust font size slice-by-slice
for autotext, value, label, wedge in zip(autotexts, values, labels, wedges):
    pct = value / total * 100

    # Smaller font for narrow slices, especially calcination-type labels
    if pct < 6:
        autotext.set_fontsize(8)
    elif str(label).lower().strip() == "calcination":
        autotext.set_fontsize(10)
    elif pct < 15:
        autotext.set_fontsize(11)
    else:
        autotext.set_fontsize(13)

    # Text contrast
    r, g, b, _ = wedge.get_facecolor()
    luminance = 0.299 * r + 0.587 * g + 0.114 * b
    autotext.set_color("black" if luminance > 0.58 else "white")

ax.axis("equal")

plt.tight_layout()

fig.savefig(
    "calcination_operation_phrase_pie_chart_report.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    "calcination_operation_phrase_pie_chart_report.pdf",
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# Section 3 — Data preparation

The training sheet for the Snowball2 pipeline is the annotation master with the
columns the extraction framework does not use removed. `title` and `paragraph_text`
are dropped (they are filled only on a paper's first row and are never read by the
training pipeline), and so are `synthesis_method` and `material_text_span`, because
the framework targets the operation / temperature / time relation only.
`ID` and `year` are filled on every row and are kept, so any event can still be
traced back to its paper. Every annotated event is kept: this step drops columns,
never rows, and applies no operation-type filter.

`0_Labeled_Synthesis_Sep_24_2026.xlsx` (`1_Training_Set_First12`)
-> `training_events_all_operations.xlsx` (`Training_events`)


In [ ]:
# 3.1 - Write the Snowball2 training sheet
#
# Drops the four columns the training pipeline never reads and keeps ID / year
# for paper-level traceability. No rows and no operation types are filtered here.

import pandas as pd

ORIGINAL_XLSX = "0_Labeled_Synthesis_Sep_24_2026.xlsx"
SOURCE_SHEET_NAME = "1_Training_Set_First12"

OUTPUT_STEM = "training_events_all_operations"
OUTPUT_XLSX = f"{OUTPUT_STEM}.xlsx"
SHEET_NAME = "Training_events"

# Read by nothing in 001, and 002/003 take document context from the gold
# workbook (0_Annotated_12_Years_Fixed_Sep_24_2026.xlsx), not from this sheet.
DROP_COLS = [
    "title",
    "paragraph_text",
    "synthesis_method",
    "material_text_span",
]

# Must be present on every row, since they replace title/paragraph_text as the
# only link back to the source paper.
TRACE_COLS = ["ID", "year"]


df = pd.read_excel(ORIGINAL_XLSX, sheet_name=SOURCE_SHEET_NAME)
df.columns = [str(c).strip() for c in df.columns]

missing_trace = [c for c in TRACE_COLS if c not in df.columns]
if missing_trace:
    raise KeyError(f"Missing traceability column(s): {missing_trace}")

unfilled_trace = [c for c in TRACE_COLS if df[c].isna().any()]
if unfilled_trace:
    raise ValueError(
        f"Traceability column(s) not filled on every row: {unfilled_trace}"
    )

dropped = [c for c in DROP_COLS if c in df.columns]
df_out = df.drop(columns=dropped)

df_out.to_excel(OUTPUT_XLSX, sheet_name=SHEET_NAME, index=False)

print(f"Saved: {OUTPUT_XLSX}  [sheet: {SHEET_NAME}]")
print(f"Rows kept: {len(df_out)} of {len(df)}   Columns: {len(df_out.columns)}")
print(f"Dropped columns: {dropped}")
print(f"Papers (unique ID): {df_out['ID'].nunique()}   Years: {df_out['year'].nunique()}")
